In [13]:
import pandas as pd

items = pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\order_items_v2.csv",sep=";")

print(items.shape)
print(items.columns.tolist())

(112650, 7)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [18]:
payments=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_order_payments_dataset.csv")
reviews=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_order_reviews_dataset.csv")
orders =pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_orders_dataset.csv")
products=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_products_dataset.csv")
customers=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_customers_dataset.csv")
sellers=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\olist_sellers_dataset.csv")
translation=pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\sqlproject\product_category_name_translation.csv")

print("Shape of payments:",payments.shape)
print("Shape of reviews:",reviews.shape)
print("Shape of orders:",orders.shape)
print("Shape of products:",products.shape)
print("Shape of customers:",customers.shape)
print("Shape of sellers:",sellers.shape)
print("Shape of translation:",translation.shape)

Shape of payments: (103886, 5)
Shape of reviews: (99224, 7)
Shape of orders: (99441, 8)
Shape of products: (32951, 9)
Shape of customers: (99441, 5)
Shape of sellers: (3095, 4)
Shape of translation: (71, 2)


In [19]:
orders.info()
print(orders.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2

In [20]:
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print(orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts())

shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: order_status, dtype: int64


In [21]:
orders_1yr = orders[(orders["order_purchase_timestamp"] >= "2017-09-01") &
                    (orders["order_purchase_timestamp"] < "2018-09-01")].copy()

print(orders_1yr.shape)

(76124, 8)


In [22]:
mask = ((orders_1yr["order_status"] == "delivered") &
        orders_1yr["order_delivered_customer_date"].notna())

delivered_day = orders_1yr.loc[mask, "order_delivered_customer_date"].dt.normalize()
promised_day = orders_1yr.loc[mask, "order_estimated_delivery_date"].dt.normalize()

orders_1yr.loc[mask, "is_late"] = (delivered_day > promised_day).astype(int)

print(orders_1yr["is_late"].value_counts(dropna=False))

0.0    68444
1.0     5762
NaN     1918
Name: is_late, dtype: int64


In [23]:
labels = orders_1yr["is_late"].map({0: "On time", 1: "Late"}).fillna("Not counted")

summary = labels.value_counts().rename_axis("Delivery result").reset_index(name="Orders")
summary

,Delivery result,Orders
0,On time,68444
1,Late,5762
2,Not counted,1918


In [24]:
summary["Total orders"] = len(orders_1yr)
summary["Percentage"] = (summary["Orders"] / summary["Total orders"] * 100).round(2)

summary = summary[["Delivery result", "Total orders", "Orders", "Percentage"]]
summary

,Delivery result,Total orders,Orders,Percentage
0,On time,76124,68444,89.91
1,Late,76124,5762,7.57
2,Not counted,76124,1918,2.52


In [25]:
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
latest = reviews.sort_values("review_answer_timestamp").drop_duplicates("order_id", keep="last")

orders_review = orders_1yr.merge(latest[["order_id", "review_score"]], on="order_id", how="left")

counted = orders_review.dropna(subset=["is_late", "review_score"]).copy()
counted["Delivery result"] = counted["is_late"].map({0: "On time", 1: "Late"})
counted["low"] = counted["review_score"] <= 2

result = counted.groupby("Delivery result")["low"].agg(["size", "sum", "mean"]).reset_index()
result.columns = ["Delivery result", "Reviewed orders", "Low reviews", "Low review %"]
result["Low review %"] = (result["Low review %"] * 100).round(2)
result

,Delivery result,Reviewed orders,Low reviews,Low review %
0,Late,5638,3553,63.02
1,On time,68086,6363,9.35


Finding: 63.02% of late orders got low reviews vs 9.35% of on-time orders, about 6.7 times higher. Late delivery is strongly linked to bad reviews. 🎯

Next, check whether the seller handed over late:

In [26]:
late = orders_1yr.loc[orders_1yr["is_late"] == 1, ["order_id", "order_delivered_carrier_date"]]
check = late.merge(items[["order_id", "shipping_limit_date"]], on="order_id", how="left")
check["shipping_limit_date"] = pd.to_datetime(check["shipping_limit_date"])
check = check.dropna()

check["seller_late"] = check["order_delivered_carrier_date"] > check["shipping_limit_date"]
seller = check.groupby("order_id")["seller_late"].any()

table = seller.map({True: "Seller late", False: "Seller on time"}).value_counts()
table = table.rename_axis("Handover").reset_index(name="Orders")
table["Total late orders checked"] = len(seller)
table["Percentage"] = (table["Orders"] / len(seller) * 100).round(2)
table[["Handover", "Total late orders checked", "Orders", "Percentage"]]

,Handover,Total late orders checked,Orders,Percentage
0,Seller on time,5761,4183,72.61
1,Seller late,5761,1578,27.39


In [27]:
state = orders_1yr.dropna(subset=["is_late"]).merge(
    customers[["customer_id", "customer_state"]], on="customer_id", how="left")

table = state.groupby("customer_state")["is_late"].agg(["size", "sum", "mean"]).reset_index()
table.columns = ["State", "Delivered orders", "Late orders", "Late %"]
table["Late orders"] = table["Late orders"].astype(int)
table["Late %"] = (table["Late %"] * 100).round(2)

table[table["Delivered orders"] >= 30].sort_values("Late %", ascending=False).head(10)

,State,Delivered orders,Late orders,Late %
9,MA,528,117,22.16
1,AL,283,59,20.85
24,SE,230,42,18.26
16,PI,357,62,17.37
5,CE,952,159,16.70
18,RJ,9345,1388,14.85
13,PA,673,97,14.41
4,BA,2482,347,13.98
7,ES,1516,194,12.80
26,TO,204,25,12.25


Finding: MA has the highest late rate (22.16%), but RJ has the most late orders (1,388), about 24% of all late orders, so look at RJ first.

Now check the money involved:

In [28]:
pay = payments.groupby("order_id", as_index=False)["payment_value"].sum()
money = orders_1yr.merge(pay, on="order_id", how="left")

money["Group"] = money["is_late"].map({0: "On time", 1: "Late"}).fillna("Other")
money.loc[money["order_status"] == "canceled", "Group"] = "Cancelled"

table = money.groupby("Group")["payment_value"].agg(["size", "sum"]).reset_index()
table.columns = ["Group", "Orders", "Order value (R$)"]
total = money["payment_value"].sum()
table["Total value (R$)"] = round(total, 2)
table["Value %"] = (table["Order value (R$)"] / total * 100).round(2)
table.round(2)

,Group,Orders,Order value (R$),Total value (R$),Value %
0,Cancelled,409,91921.30,12275458.45,0.75
1,Late,5762,1001189.27,12275458.45,8.16
2,On time,68444,10899638.49,12275458.45,88.79
3,Other,1509,282709.39,12275458.45,2.30


Finding: late orders hold R$1.00 million (8.16%) of order value, and cancelled orders 0.75%. This is value affected, not proven lost revenue.

Now find the high-risk sellers:

In [29]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
per_order = items.groupby("order_id", as_index=False).agg(
    seller_id=("seller_id", "first"), sellers=("seller_id", "nunique"),
    limit=("shipping_limit_date", "min"))

df = orders_review.dropna(subset=["is_late"]).merge(per_order[per_order["sellers"] == 1], on="order_id")
df["low"] = (df["review_score"] <= 2).astype(float).where(df["review_score"].notna())
df["handover"] = (df["order_delivered_carrier_date"] > df["limit"]).astype(float).where(df["order_delivered_carrier_date"].notna())

table = df.groupby("seller_id").agg(Orders=("order_id", "count"), Low=("low", "mean"), Late=("handover", "mean")).reset_index()
table = table[table["Orders"] >= 30]
risk = table[(table["Low"] > 1.5 * df["low"].mean()) | (table["Late"] > 1.5 * df["handover"].mean())].copy()

print("Sellers checked:", len(table), "| High-risk:", len(risk))
risk["Low review %"] = (risk["Low"] * 100).round(2)
risk["Late handover %"] = (risk["Late"] * 100).round(2)
risk[["seller_id", "Orders", "Low review %", "Late handover %"]].sort_values("Low review %", ascending=False).head(10)

Sellers checked: 510 | High-risk: 146


,seller_id,Orders,Low review %,Late handover %
291,1ca7077d890b907f89be8c954a02686a,106,60.00,51.89
487,2eb70248d66e0e3ef83659f71b244378,185,47.80,32.43
1696,a49928bcdf77c55c6d6e05e09a9b4ca5,74,43.24,68.92
1176,710e3548e02bc1d2831dfc4f1b5b14d4,77,42.86,27.27
1562,972d0f9cf61b499a4812cf0bfa3ad3c4,75,41.10,34.67
1178,712e6ed8aa4aa1fa65dab41fed5737e4,61,40.00,22.95
886,54965bbe3e4f07ae045b90b0b8541f52,72,39.71,54.17
1908,bbad7e518d7af88a0897397ffdca1979,53,38.46,39.62
1158,6fd52c528dcb38be2eea044946b811f8,64,38.10,29.69
1009,602044f2c16190c2c6e45eb35c2e21cb,45,36.36,0.00


Finding: the top seller by low reviews has 60% low reviews and 51.89% late handovers, so check it first. The tenth has 36.36% low reviews but no late handovers, so look for other causes.

Now check product categories:

In [30]:
cat = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
cat = cat.merge(translation, on="product_category_name", how="left")
cat["Category"] = cat["product_category_name_english"].fillna(cat["product_category_name"]).fillna("unknown")

per_cat = cat.groupby("order_id").agg(Category=("Category", "first"), n=("Category", "nunique")).reset_index()

df = orders_review.merge(per_cat[per_cat["n"] == 1], on="order_id")
df = df.dropna(subset=["is_late", "review_score"])
df["low"] = df["review_score"] <= 2

table = df.groupby("Category").agg(Orders=("order_id", "count"), Low=("low", "mean"), Late=("is_late", "mean")).reset_index()
table = table[table["Orders"] >= 30]
table["Low review %"] = (table["Low"] * 100).round(2)
table["Late %"] = (table["Late"] * 100).round(2)

table[["Category", "Orders", "Low review %", "Late %"]].sort_values("Low review %", ascending=False).head(10)

,Category,Orders,Low review %,Late %
57,office_furniture,833,23.41,9.60
4,audio,271,22.14,13.28
34,fixed_telephony,106,21.70,6.60
30,fashion_male_clothing,55,20.00,3.64
41,furniture_mattress_and_upholstery,30,20.00,16.67
33,fashion_underwear_beach,76,19.74,11.84
72,unknown,914,17.51,8.64
7,bed_bath_table,6812,16.68,8.84
58,party_supplies,36,16.67,0.00
19,construction_tools_safety,149,15.44,2.68


Finding: among single-category orders, office_furniture has the highest low review rate (23.41%), but bed_bath_table has far more low reviews (about 1,136).

Now check whether late delivery affects repeat orders:

In [31]:
df = orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
first = df.sort_values("order_purchase_timestamp").drop_duplicates("customer_unique_id")
first = first.merge(orders_1yr[["order_id", "is_late"]], on="order_id").dropna(subset=["is_late"])

later = df.merge(first[["customer_unique_id", "order_delivered_customer_date"]],
                 on="customer_unique_id", suffixes=("", "_first"))
back = later.loc[later["order_purchase_timestamp"] > later["order_delivered_customer_date_first"],
                 "customer_unique_id"]

first["returned"] = first["customer_unique_id"].isin(back)
first["Delivery result"] = first["is_late"].map({0: "On time", 1: "Late"})

table = first.groupby("Delivery result")["returned"].agg(["size", "sum", "mean"]).reset_index()
table.columns = ["Delivery result", "First-time customers", "Ordered again", "Repeat %"]
table["Repeat %"] = (table["Repeat %"] * 100).round(2)
table

,Delivery result,First-time customers,Ordered again,Repeat %
0,Late,5594,63,1.13
1,On time,66019,1015,1.54


Finding: 1.13% of late-delivery customers ordered again vs 1.54% of on-time customers. Late delivery is linked to fewer repeat orders, but very few customers return in either group.

Now check the monthly trend:

In [32]:
df = orders_1yr.dropna(subset=["is_late"]).copy()
df["Month"] = df["order_purchase_timestamp"].dt.strftime("%Y-%m")

table = df.groupby("Month")["is_late"].agg(["size", "sum", "mean"]).reset_index()
table.columns = ["Month", "Delivered orders", "Late orders", "Late %"]
table["Late orders"] = table["Late orders"].astype(int)
table["Late %"] = (table["Late %"] * 100).round(2)
table

,Month,Delivered orders,Late orders,Late %
0,2017-09,4150,182,4.39
1,2017-10,4478,187,4.18
2,2017-11,7288,904,12.40
3,2017-12,5513,411,7.46
4,2018-01,7069,403,5.70
5,2018-02,6555,926,14.13
6,2018-03,7003,1328,18.96
7,2018-04,6798,306,4.50
8,2018-05,6749,443,6.56
9,2018-06,6096,71,1.16


Finding: late rates peaked in March 2018 (18.96%). Nov 2017, Feb 2018 and Mar 2018 together had about 55% of all late orders.

Now create your Tableau file (one row per order):

In [33]:
folder = r"C:\Users\Admin\OneDrive\Desktop\sqlproject"

final = (orders_review
         .merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id", how="left")
         .merge(pay, on="order_id", how="left"))

final["delivery_result"] = final["is_late"].map({0: "On time", 1: "Late"}).fillna("Not counted")
final["low_review"] = (final["review_score"] <= 2).astype(float).where(final["review_score"].notna())

print(final.shape, final["order_id"].is_unique)
final.to_csv(folder + r"\tableau_orders.csv", index=False)

(76124, 15) True


In [34]:
# Seller file
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"])
per_order = items.groupby("order_id", as_index=False).agg(
    seller_id=("seller_id", "first"), seller_count=("seller_id", "nunique"),
    limit=("shipping_limit_date", "min"))

seller_file = final.dropna(subset=["is_late"]).merge(
    per_order[per_order["seller_count"] == 1], on="order_id")
carrier = seller_file["order_delivered_carrier_date"]
seller_file["late_handover"] = (carrier > seller_file["limit"]).astype(float).where(carrier.notna())
seller_file = seller_file[["order_id", "seller_id", "low_review", "late_handover"]]
seller_file.to_csv(folder + r"\tableau_sellers.csv", index=False)

# Category file
cat = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
cat = cat.merge(translation, on="product_category_name", how="left")
cat["category"] = cat["product_category_name_english"].fillna(cat["product_category_name"]).fillna("unknown")
per_cat = cat.groupby("order_id").agg(
    category=("category", "first"), n=("category", "nunique")).reset_index()

cat_file = final.dropna(subset=["is_late", "review_score"]).merge(per_cat[per_cat["n"] == 1], on="order_id")
cat_file = cat_file[["order_id", "category", "low_review"]]
cat_file.to_csv(folder + r"\tableau_categories.csv", index=False)

print("Sellers:", seller_file.shape)
print("Categories:", cat_file.shape)

Sellers: (73152, 4)
Categories: (73120, 3)


In [35]:
import numpy as np

cancel = orders_1yr[orders_1yr["order_status"] == "canceled"]

stage = np.select(
    [cancel["order_delivered_customer_date"].notna(),
     cancel["order_delivered_carrier_date"].notna(),
     cancel["order_approved_at"].notna()],
    ["Reached customer", "Given to carrier", "Approved, not shipped"],
    default="Not approved")

table = pd.Series(stage).value_counts().rename_axis("Stage reached").reset_index(name="Cancelled orders")
table["Total cancelled"] = len(cancel)
table["Percentage"] = (table["Cancelled orders"] / len(cancel) * 100).round(2)

print("Cancellation rate:", round(len(cancel) / len(orders_1yr) * 100, 2), "%")
table

Cancellation rate: 0.54 %


,Stage reached,Cancelled orders,Total cancelled,Percentage
0,"Approved, not shipped",256,409,62.59
1,Not approved,83,409,20.29
2,Given to carrier,69,409,16.87
3,Reached customer,1,409,0.24


In [36]:
tables = {
    "orders": (orders, ["order_id"]),
    "items": (items, ["order_id", "order_item_id"]),
    "payments": (payments, ["order_id", "payment_sequential"]),
    "reviews": (reviews, ["order_id", "review_id"]),
    "customers": (customers, ["customer_id"]),
    "products": (products, ["product_id"]),
    "sellers": (sellers, ["seller_id"]),
    "translation": (translation, ["product_category_name"])
}

rows = []
for name, (df, key) in tables.items():
    missing = df.isna().sum()
    columns = ", ".join(f"{c}: {n}" for c, n in missing[missing > 0].items())
    rows.append([name, len(df), df.duplicated(key).sum(), columns or "None"])

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows, columns=["Table", "Rows", "Duplicate keys", "Missing values"])

,Table,Rows,Duplicate keys,Missing values
0,orders,99441,0,"order_approved_at: 160, order_delivered_carrier_date: 1783, order_delivered_customer_date: 2965"
1,items,112650,0,None
2,payments,103886,0,None
3,reviews,99224,0,"review_comment_title: 87656, review_comment_message: 58247"
4,customers,99441,0,None
5,products,32951,0,"product_category_name: 610, product_name_lenght: 610, product_description_lenght: 610, product_photos_qty: 610, product_weight_g: 2, product_length_cm: 2, product_height_cm: 2, product_width_cm: 2"
6,sellers,3095,0,None
7,translation,71,0,None
